# Section 1: Split and baseline

In [1]:
import sys
sys.path.append('../src')

from data import load_hour_data, chronological_split, get_xy

In [2]:
df = load_hour_data()

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (17379, 17)

First 5 rows:


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [3]:
train, test = chronological_split(df)

print("Training rows:", len(train))
print("Testing rows:", len(test))

print("\nTraining period:")
print("From:", train['dteday'].min())
print("To:", train['dteday'].max())

print("\nTesting period:")
print("From:", test['dteday'].min())
print("To:", test['dteday'].max())

Training rows: 13003
Testing rows: 4376

Training period:
From: 2011-01-01 00:00:00
To: 2012-06-30 00:00:00

Testing period:
From: 2012-07-01 00:00:00
To: 2012-12-31 00:00:00


### Why use a chronological split?

This dataset contains bike rental observations over time, so the model should learn from past data and predict future data. A random split could place future observations in the training set while earlier observations are in the test set, which would not represent a real forecasting situation. Therefore, the data is split chronologically at 2012-07-01, with earlier observations used for training and later observations used for testing.

In [4]:
X_train, y_train = get_xy(train)
X_test, y_test = get_xy(test)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nX_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (13003, 12)
y_train shape: (13003,)

X_test shape: (4376, 12)
y_test shape: (4376,)


### Baseline Linear Regression

For the baseline model, I use four simple features: hour of day (`hr`), temperature (`temp`), whether it is a working day (`workingday`), and year (`yr`). This gives us a simple starting model that later models can be compared against.

In [5]:
baseline_features = ['hr', 'temp', 'workingday', 'yr']

X_train_baseline = X_train[baseline_features]
X_test_baseline = X_test[baseline_features]

display(X_train_baseline.head())

,hr,temp,workingday,yr
0,0,0.24,0,0
1,1,0.22,0,0
2,2,0.22,0,0
3,3,0.24,0,0
4,4,0.24,0,0


In [6]:
from sklearn.linear_model import LinearRegression

baseline_model = LinearRegression()

baseline_model.fit(X_train_baseline, y_train)

baseline_predictions = baseline_model.predict(X_test_baseline)

print("Baseline model trained successfully.")
print("Number of predictions:", len(baseline_predictions))

Baseline model trained successfully.
Number of predictions: 4376


/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
mae = mean_absolute_error(y_test, baseline_predictions)
r2 = r2_score(y_test, baseline_predictions)

print("Baseline Linear Regression Results")
print("----------------------------------")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

Baseline Linear Regression Results
----------------------------------
RMSE: 191.29796121072502
MAE : 145.32068901300266
R2  : 0.2417854955151757


### Random Split Demo — NOT the headline result

The chronological split above is the main evaluation because this is time-based data. The following random split is only a demonstration to compare how the evaluation changes when observations from different time periods are mixed.

In [8]:
from sklearn.model_selection import train_test_split

X_random = X_train[baseline_features]
y_random = y_train

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X_random,
    y_random,
    test_size=0.3,
    random_state=42
)

random_model = LinearRegression()
random_model.fit(X_train_random, y_train_random)

random_predictions = random_model.predict(X_test_random)

random_rmse = np.sqrt(
    mean_squared_error(y_test_random, random_predictions)
)
random_mae = mean_absolute_error(y_test_random, random_predictions)
random_r2 = r2_score(y_test_random, random_predictions)

print("Random Split Demo Results")
print("-------------------------")
print("RMSE:", random_rmse)
print("MAE :", random_mae)
print("R2  :", random_r2)

Random Split Demo Results
-------------------------
RMSE: 128.22980430638518
MAE : 97.17559131135569
R2  : 0.34605241855478674


/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


### Chronological vs Random Split

The random split gives better scores than the chronological split. The random split has an RMSE of about 128.23 compared with 191.30 for the chronological split, while its R² is about 0.346 compared with 0.242.

However, the chronological split is the main result because this dataset is time-based. A random split mixes observations from different time periods, so the model can learn from data that would not be available when making a real future prediction. Therefore, the random split can make the model performance look better than it would in a realistic forecasting situation.

In [9]:
import pandas as pd

baseline_results = pd.DataFrame({
    'dteday': test['dteday'].values,
    'hr': test['hr'].values,
    'y_true': y_test.values,
    'baseline': baseline_predictions
})

baseline_results.to_csv(
    '../results/preds_baseline.csv',
    index=False
)

display(baseline_results.head())

print("Saved results/preds_baseline.csv successfully.")

,dteday,hr,y_true,baseline
0,2012-07-01,0,149,218.190681
1,2012-07-01,1,93,219.785255
2,2012-07-01,2,90,221.379829
3,2012-07-01,3,33,229.374799
4,2012-07-01,4,4,230.969373


Saved results/preds_baseline.csv successfully.


### Limitation of the Baseline Model

The baseline Linear Regression model is simple and cannot capture all the patterns in bike demand. In particular, it treats `hr` as a normal numerical feature and assumes a simple linear relationship between hour and demand. However, bike demand can have multiple peaks during a day, such as morning and evening commuting hours. A single linear relationship with `hr` cannot represent these two peaks properly. This is one reason why the baseline model has relatively high prediction error.

# Section 2: Linear, Ridge, Lasso and Random Forest

In this section, we train and compare four regression models using the final preprocessing pipeline. Hyperparameters are selected using TimeSeriesSplit on the training data only, while the test set is reserved for final evaluation.

In [20]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd

from data import load_hour_data, chronological_split, get_xy
from preprocessing import build_preprocessor

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = load_hour_data()

train, test = chronological_split(df)

X_train, y_train = get_xy(train)
X_test, y_test = get_xy(test)

tscv = TimeSeriesSplit(n_splits=5)

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Setup complete.")

Train rows: 13003
Test rows: 4376
Setup complete.


## 2.1 Linear Regression

First, we train a Linear Regression model using the final preprocessing pipeline. The preprocessing is fitted only on the training data to avoid data leakage. We then evaluate the fitted model on the held-out test period using RMSE, MAE and R².

In [11]:
linear_pipe = Pipeline([
    ('prep', build_preprocessor()),
    ('model', LinearRegression())
])

# Fit only on training data
linear_pipe.fit(X_train, y_train)

# Predict the test period
linear_pred = linear_pipe.predict(X_test)

# Evaluation metrics
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_mae = mean_absolute_error(y_test, linear_pred)
linear_r2 = r2_score(y_test, linear_pred)

print(f"Linear Regression RMSE: {linear_rmse:.2f}")
print(f"Linear Regression MAE:  {linear_mae:.2f}")
print(f"Linear Regression R2:   {linear_r2:.3f}")

Linear Regression RMSE: 165.73
Linear Regression MAE:  123.92
Linear Regression R2:   0.431


/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


## 2.2 Ridge Regression

Ridge Regression adds L2 regularisation to Linear Regression, which helps control large coefficients and can improve stability when features are correlated.

The regularisation strength (`alpha`) is selected using GridSearchCV with TimeSeriesSplit on the training data only. We do not choose alpha using the test set because the test period must remain unseen until final evaluation. TimeSeriesSplit is used instead of shuffled K-fold cross-validation because this is time-ordered data and future observations should not be used to predict earlier observations.

In [12]:
ridge_pipe = Pipeline([
    ('prep', build_preprocessor()),
    ('model', Ridge())
])

ridge_param_grid = {
    'model__alpha': np.logspace(-3, 3, 13)
}

ridge_grid = GridSearchCV(
    ridge_pipe,
    ridge_param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

print("Best Ridge alpha:", ridge_grid.best_params_['model__alpha'])

# Best model is automatically refitted on the full training data
ridge_best = ridge_grid.best_estimator_

ridge_pred = ridge_best.predict(X_test)

ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_r2 = r2_score(y_test, ridge_pred)

print(f"Ridge Regression RMSE: {ridge_rmse:.2f}")
print(f"Ridge Regression MAE:  {ridge_mae:.2f}")
print(f"Ridge Regression R2:   {ridge_r2:.3f}")

Best Ridge alpha: 0.001
Ridge Regression RMSE: 165.73
Ridge Regression MAE:  123.92
Ridge Regression R2:   0.431


/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ 

## 2.3 Lasso Regression

Lasso Regression uses L1 regularisation. Unlike Ridge, Lasso can shrink some coefficients exactly to zero, so it can also perform a form of feature selection.

The regularisation strength is selected using GridSearchCV with the same TimeSeriesSplit on the training data only.

In [13]:
lasso_pipe = Pipeline([
    ('prep', build_preprocessor()),
    ('model', Lasso(max_iter=10000))
])

lasso_param_grid = {
    'model__alpha': np.logspace(-4, 1, 12)
}

lasso_grid = GridSearchCV(
    lasso_pipe,
    lasso_param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

lasso_grid.fit(X_train, y_train)

print("Best Lasso alpha:", lasso_grid.best_params_['model__alpha'])

lasso_best = lasso_grid.best_estimator_
lasso_pred = lasso_best.predict(X_test)

lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))
lasso_mae = mean_absolute_error(y_test, lasso_pred)
lasso_r2 = r2_score(y_test, lasso_pred)

print(f"Lasso Regression RMSE: {lasso_rmse:.2f}")
print(f"Lasso Regression MAE:  {lasso_mae:.2f}")
print(f"Lasso Regression R2:   {lasso_r2:.3f}")

/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_m

Best Lasso alpha: 0.0001
Lasso Regression RMSE: 165.73
Lasso Regression MAE:  123.92
Lasso Regression R2:   0.431


/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tanisharoy/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [14]:
# Get feature names after preprocessing
lasso_feature_names = lasso_best.named_steps['prep'].get_feature_names_out()

# Get Lasso coefficients
lasso_coefficients = lasso_best.named_steps['model'].coef_

# Find coefficients that became exactly zero
zero_mask = lasso_coefficients == 0
zero_features = lasso_feature_names[zero_mask]

print("Total coefficients:", len(lasso_coefficients))
print("Number of coefficients exactly zero:", zero_mask.sum())

print("\nFeatures with zero coefficients:")
for feature in zero_features:
    print(feature)

Total coefficients: 26
Number of coefficients exactly zero: 3

Features with zero coefficients:
cat__season_2
cat__weekday_2
bin__workingday


## 2.4 Random Forest Regression

Random Forest is a non-linear ensemble model that combines predictions from multiple decision trees. Unlike the linear models, it can naturally capture non-linear relationships and interactions between features, such as the relationship between hour of day and working-day status.

The model is tuned using TimeSeriesSplit on the training data only. We compare different numbers of trees, maximum tree depths and minimum leaf sizes.

In [15]:
forest_pipe = Pipeline([
    ('prep', build_preprocessor()),
    ('model', RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

forest_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None],
    'model__min_samples_leaf': [1, 5]
}

forest_grid = GridSearchCV(
    forest_pipe,
    forest_param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

forest_grid.fit(X_train, y_train)

print("Best Random Forest parameters:")
print(forest_grid.best_params_)

forest_best = forest_grid.best_estimator_

forest_pred = forest_best.predict(X_test)

forest_rmse = np.sqrt(mean_squared_error(y_test, forest_pred))
forest_mae = mean_absolute_error(y_test, forest_pred)
forest_r2 = r2_score(y_test, forest_pred)

print(f"\nRandom Forest RMSE: {forest_rmse:.2f}")
print(f"Random Forest MAE:  {forest_mae:.2f}")
print(f"Random Forest R2:   {forest_r2:.3f}")

Best Random Forest parameters:
{'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}

Random Forest RMSE: 83.69
Random Forest MAE:  56.37
Random Forest R2:   0.855


## 2.5 Save Model Predictions and Best Parameters

The predictions from all four models are saved for the evaluation stage. The selected hyperparameters are also saved so that the final evaluation can reproduce the chosen models.

In [16]:
preds_models = pd.DataFrame({
    'dteday': test['dteday'].values,
    'hr': test['hr'].values,
    'y_true': y_test.values,
    'linear': linear_pred,
    'ridge': ridge_pred,
    'lasso': lasso_pred,
    'forest': forest_pred
})

preds_models.to_csv('../results/preds_models.csv', index=False)

print("Saved ../results/preds_models.csv")
print("Shape:", preds_models.shape)

preds_models.head()

Saved ../results/preds_models.csv
Shape: (4376, 7)


,dteday,hr,y_true,linear,ridge,lasso,forest
0,2012-07-01,0,149,205.564529,205.564511,205.565497,146.510
1,2012-07-01,1,93,177.371181,177.371163,177.372145,110.890
2,2012-07-01,2,90,164.470219,164.470195,164.471271,75.965
3,2012-07-01,3,33,158.393615,158.393578,158.394699,35.700
4,2012-07-01,4,4,156.180915,156.180886,156.181862,8.790


In [17]:
best_params = pd.DataFrame([
    {
        'model': 'ridge',
        'param': 'alpha',
        'value': str(ridge_grid.best_params_['model__alpha'])
    },
    {
        'model': 'lasso',
        'param': 'alpha',
        'value': str(lasso_grid.best_params_['model__alpha'])
    },
    {
        'model': 'forest',
        'param': 'n_estimators',
        'value': str(forest_grid.best_params_['model__n_estimators'])
    },
    {
        'model': 'forest',
        'param': 'max_depth',
        'value': str(forest_grid.best_params_['model__max_depth'])
    },
    {
        'model': 'forest',
        'param': 'min_samples_leaf',
        'value': str(forest_grid.best_params_['model__min_samples_leaf'])
    }
])

best_params.to_csv('../results/best_params.csv', index=False)

print("Saved ../results/best_params.csv")
best_params

Saved ../results/best_params.csv


,model,param,value
0,ridge,alpha,0.001
1,lasso,alpha,0.0001
2,forest,n_estimators,200
3,forest,max_depth,None
4,forest,min_samples_leaf,1


## 2.6 Final Model Comparison

The four models are compared below using RMSE on the same held-out test period. Lower RMSE indicates better predictive accuracy.

In [18]:
model_comparison = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Ridge Regression',
        'Lasso Regression',
        'Random Forest'
    ],
    'Test RMSE': [
        linear_rmse,
        ridge_rmse,
        lasso_rmse,
        forest_rmse
    ]
})

model_comparison = model_comparison.sort_values('Test RMSE').reset_index(drop=True)

model_comparison

,Model,Test RMSE
0,Random Forest,83.688666
1,Linear Regression,165.733490
2,Ridge Regression,165.733497
3,Lasso Regression,165.733642


### Model Comparison Findings

Random Forest achieved the best performance with a test RMSE of approximately 83.69, substantially lower than the RMSE of approximately 165.73 obtained by Linear Regression, Ridge and Lasso.

The three linear models produced almost identical test performance. Ridge selected an alpha of 0.001, while Lasso selected an alpha of 0.0001. Lasso reduced 3 of the 26 coefficients exactly to zero: `season_2`, `weekday_2` and `workingday`.

Random Forest performs much better because bike-rental demand contains non-linear relationships and interactions that the linear models cannot represent as effectively. In particular, demand patterns depend strongly on interactions such as hour of day and whether the day is a working day.

Therefore, among the four models tested here, Random Forest is the strongest model for predicting hourly bike-rental demand on the held-out test period.